### ANN Model creation



In [170]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [171]:
df=pd.read_csv('../Datasets/traffic_dataset.csv')
df.head()

,vehicles_N,vehicles_S,vehicles_E,vehicles_W,ped_N,ped_S,ped_E,ped_W,emg_N,emg_S,emg_E,emg_W,green_N,green_S,green_E,green_W
0,270,317,274,291,1,2,1,3,2,3,1,0,485.5,570.6,493.7,528.3
1,35,58,41,271,1,3,3,3,2,1,2,2,62.5,107.9,76.3,490.3
2,260,379,340,305,0,0,1,3,1,3,2,0,467.0,679.2,611.5,553.5
3,24,52,48,381,3,2,0,3,2,1,2,3,45.7,95.6,84.4,687.3
4,286,79,267,64,1,3,1,3,3,3,1,3,513.3,143.7,481.1,116.7


In [172]:
df.isnull().sum()

vehicles_N    0
vehicles_S    0
vehicles_E    0
vehicles_W    0
ped_N         0
ped_S         0
ped_E         0
ped_W         0
emg_N         0
emg_S         0
emg_E         0
emg_W         0
green_N       0
green_S       0
green_E       0
green_W       0
dtype: int64

In [173]:
df.duplicated().sum()

0

In [174]:
df.head()

,vehicles_N,vehicles_S,vehicles_E,vehicles_W,ped_N,ped_S,ped_E,ped_W,emg_N,emg_S,emg_E,emg_W,green_N,green_S,green_E,green_W
0,270,317,274,291,1,2,1,3,2,3,1,0,485.5,570.6,493.7,528.3
1,35,58,41,271,1,3,3,3,2,1,2,2,62.5,107.9,76.3,490.3
2,260,379,340,305,0,0,1,3,1,3,2,0,467.0,679.2,611.5,553.5
3,24,52,48,381,3,2,0,3,2,1,2,3,45.7,95.6,84.4,687.3
4,286,79,267,64,1,3,1,3,3,3,1,3,513.3,143.7,481.1,116.7


In [176]:
X=df.drop(columns=["green_N","green_S","green_E","green_W"])
Y=df[["green_N","green_S","green_E","green_W"]]

In [177]:
X

,vehicles_N,vehicles_S,vehicles_E,vehicles_W,ped_N,ped_S,ped_E,ped_W,emg_N,emg_S,emg_E,emg_W
0,270,317,274,291,1,2,1,3,2,3,1,0
1,35,58,41,271,1,3,3,3,2,1,2,2
2,260,379,340,305,0,0,1,3,1,3,2,0
3,24,52,48,381,3,2,0,3,2,1,2,3
4,286,79,267,64,1,3,1,3,3,3,1,3
...,...,...,...,...,...,...,...,...,...,...,...,...
14995,32,3,32,35,2,3,2,3,3,0,3,1
14996,78,191,50,20,3,2,3,0,2,2,3,2
14997,32,15,174,104,3,2,1,0,1,0,0,2
14998,58,45,23,49,1,2,2,0,0,2,3,1


In [178]:
Y

,green_N,green_S,green_E,green_W
0,485.5,570.6,493.7,528.3
1,62.5,107.9,76.3,490.3
2,467.0,679.2,611.5,553.5
3,45.7,95.6,84.4,687.3
4,513.3,143.7,481.1,116.7
...,...,...,...,...
14995,57.6,9.9,57.6,66.5
14996,142.9,344.8,91.5,34.0
14997,61.1,30.0,314.7,185.2
14998,105.9,82.0,41.4,87.2


In [179]:
# split test and training data
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=.2,random_state=15)

In [180]:
# stanrddization data for independent values
from sklearn.preprocessing import StandardScaler

scalar_independent=StandardScaler()
X_train=scalar_independent.fit_transform(X_train)
X_test=scalar_independent.transform(X_test)

In [181]:
# standardization data for dependent values

scalar_dependent=StandardScaler()
Y_train=scalar_dependent.fit_transform(Y_train)
Y_test=scalar_dependent.transform(Y_test)

In [182]:
print(Y_train.min(), Y_train.max(), Y_train.std())


-0.9949527606106421 5.126918539010427 1.0


In [183]:
# save StandardScaler models

import pickle

with open("../models/traffic_models/scaler_independent.pkl","wb") as file:
	pickle.dump(scalar_independent,file)

with open("../models/traffic_models/scaler_dependent.pkl","wb") as file:
	pickle.dump(scalar_dependent,file)

### ANN Training

In [184]:
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime
from tensorflow.keras.layers import Dropout
from tensorflow.keras.regularizers import l2

In [185]:
# Model creation
traffic_model = Sequential([
    Dense(64, input_dim=X_train.shape[1], activation='relu'),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(4, activation='linear')
])

In [186]:
traffic_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,668 (69.02 KB)

 Trainable params: 17,668 (69.02 KB)

 Non-trainable params: 0 (0.00 B)

In [187]:
optimiser=tensorflow.keras.optimizers.Adam(learning_rate=0.05)
loss=tensorflow.keras.losses.mse

In [188]:
# set optimisers , loss functions
traffic_model.compile(optimizer=optimiser,loss=loss,metrics=['mae'])

In [189]:
# set up early stoping 
# early_stopping_call_back=EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)


early_stopping_call_back = EarlyStopping(
    monitor='val_loss',
    patience=15,              # Try lower patience for faster stopping
   
    restore_best_weights=True,
    verbose=1                 # Print a message when stopping
)

In [190]:
# set up log files

log_dir="../logs/morning_time_log/fit/"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorBoard_callback=TensorBoard(log_dir=log_dir, histogram_freq=1)

In [191]:
# Store the training history for plotting
history = traffic_model.fit(
    X_train,
    Y_train,
    epochs=1000,
    validation_data=(X_test, Y_test),
    callbacks=[tensorBoard_callback, early_stopping_call_back],
    verbose=1
)

Epoch 1/1000
375/375 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.6268 - mae: 0.4155 - val_loss: 0.0164 - val_mae: 0.0874
Epoch 2/1000
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0250 - mae: 0.0969 - val_loss: 0.4205 - val_mae: 0.4720
Epoch 3/1000
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.2329 - mae: 0.3289 - val_loss: 0.0087 - val_mae: 0.0685
Epoch 4/1000
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0094 - mae: 0.0637 - val_loss: 0.0069 - val_mae: 0.0611
Epoch 5/1000
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0050 - mae: 0.0471 - val_loss: 0.0020 - val_mae: 0.0316
Epoch 6/1000
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0052 - mae: 0.0418 - val_loss: 0.0041 - val_mae: 0.0465
Epoch 7/1000
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0097 - mae: 0.0575 - val_loss: 0.0299 - val_mae: 0.1298
Epoch 8/1000
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0299 - mae: 0.1189 - val_loss: 0.0091 - val_mae: 0.0687
Epoch 9/1000
375/375 ━━━━━━━━━━━━━━━━━━━

In [192]:
traffic_model.metrics

[<Mean name=loss>, <CompileMetrics name=compile_metrics>]

In [193]:
results = traffic_model.evaluate(X_test, Y_test, return_dict=True)
print(results)

94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 7.5716e-05 - mae: 0.0060
{'loss': 7.670582999708131e-05, 'mae': 0.006027513649314642}


In [194]:
df.head()

,vehicles_N,vehicles_S,vehicles_E,vehicles_W,ped_N,ped_S,ped_E,ped_W,emg_N,emg_S,emg_E,emg_W,green_N,green_S,green_E,green_W
0,270,317,274,291,1,2,1,3,2,3,1,0,485.5,570.6,493.7,528.3
1,35,58,41,271,1,3,3,3,2,1,2,2,62.5,107.9,76.3,490.3
2,260,379,340,305,0,0,1,3,1,3,2,0,467.0,679.2,611.5,553.5
3,24,52,48,381,3,2,0,3,2,1,2,3,45.7,95.6,84.4,687.3
4,286,79,267,64,1,3,1,3,3,3,1,3,513.3,143.7,481.1,116.7


In [195]:
input_data = pd.DataFrame([{
    'vehicles_N': 100,
    'vehicles_S': 110,
    'vehicles_E': 200,
    'vehicles_W': 12,
    'ped_N': 0,
    'ped_S': 0,
    'ped_E': 0,
    'ped_W': 0,
    'emg_N': 0,
    'emg_S': 0,
    'emg_E': 0,
    'emg_W': 0,
}])
input_data

,vehicles_N,vehicles_S,vehicles_E,vehicles_W,ped_N,ped_S,ped_E,ped_W,emg_N,emg_S,emg_E,emg_W
0,100,110,200,12,0,0,0,0,0,0,0,0


In [196]:
input_data_scalar=scalar_independent.transform(input_data)
input_data_scalar

array([[ 0.52662983,  0.68676496,  2.07345706, -0.78457284, -1.32766079,
        -1.35585321, -1.33829182, -1.34708579, -1.34938732, -1.35461287,
        -1.34608834, -1.3469193 ]])

In [197]:
pred=traffic_model.predict(input_data_scalar)
pred[0]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step


array([ 0.51306117,  0.6751207 ,  2.0690806 , -0.78727674], dtype=float32)

In [198]:
input_data

,vehicles_N,vehicles_S,vehicles_E,vehicles_W,ped_N,ped_S,ped_E,ped_W,emg_N,emg_S,emg_E,emg_W
0,100,110,200,12,0,0,0,0,0,0,0,0


In [199]:
normal_value=scalar_dependent.inverse_transform(pred)
normal_value

array([[179.09062 , 197.38998 , 360.31427 ,  21.986698]], dtype=float32)

In [ ]:
traffic_model.save('../models/traffic_models/traffic_model.h5')

### Model Accuracy Evaluation using R² Score

Calculate R² (coefficient of determination) to assess model accuracy. R² represents the proportion of variance in the dependent variable that is predictable from the independent variables.

**R² Formula:** R² = 1 - (SS_res / SS_tot)
- SS_res = Σ(y_true - y_pred)² (residual sum of squares)
- SS_tot = Σ(y_true - y_mean)² (total sum of squares)
- **Range:** R² ∈ [0, 1] (higher values indicate better model performance)

In [ ]:
# Import R² score calculation
from sklearn.metrics import r2_score

# Make predictions on test data
Y_pred_scaled = traffic_model.predict(X_test, verbose=0)

# Transform predictions back to original scale for R² calculation
Y_pred_original = scalar_dependent.inverse_transform(Y_pred_scaled)
Y_test_original = scalar_dependent.inverse_transform(Y_test)

print("=" * 60)
print("MODEL ACCURACY ASSESSMENT USING R² SCORE")
print("=" * 60)

# Calculate R² for each traffic direction
directions = ['green_N', 'green_S', 'green_E', 'green_W']
individual_r2_scores = {}

for i, direction in enumerate(directions):
    r2 = r2_score(Y_test_original[:, i], Y_pred_original[:, i])
    individual_r2_scores[direction] = r2
    print(f"R² Score for {direction}: {r2:.6f}")

# Calculate overall R² score (average across all outputs)
overall_r2 = r2_score(Y_test_original, Y_pred_original, multioutput='uniform_average')
print(f"\nOverall R² Score: {overall_r2:.6f}")

# Convert R² to accuracy percentage
accuracy_percentage = overall_r2 * 100
print(f"Model Accuracy (based on R²): {accuracy_percentage:.2f}%")

print("\n" + "=" * 60)
print("PERFORMANCE INTERPRETATION:")
print("=" * 60)

def interpret_r2_score(r2):
    """Provide interpretation of R² score"""
    if r2 >= 0.90:
        return "Excellent model performance"
    elif r2 >= 0.80:
        return "Good model performance"
    elif r2 >= 0.70:
        return "Acceptable model performance"
    elif r2 >= 0.60:
        return "Moderate model performance"
    else:
        return "Poor model performance - consider model improvement"

for direction, r2 in individual_r2_scores.items():
    interpretation = interpret_r2_score(r2)
    print(f"{direction}: {interpretation} (R² = {r2:.6f})")

print(f"\nOverall Model: {interpret_r2_score(overall_r2)} (R² = {overall_r2:.6f})")
print("\nNote: R² values closer to 1.0 indicate better model performance.")
print("R² represents the proportion of variance explained by the model.")

In [ ]:
# Compare current evaluation metrics
print("\n" + "=" * 60)
print("COMPREHENSIVE MODEL EVALUATION SUMMARY")
print("=" * 60)

# Current MAE from model evaluation
current_mae = results['mae']
current_loss = results['loss']

print(f"Mean Absolute Error (MAE): {current_mae:.6f}")
print(f"Loss (MSE): {current_loss:.6f}")
print(f"R² Score: {overall_r2:.6f}")
print(f"Model Accuracy: {accuracy_percentage:.2f}%")

print("\nMETRIC INTERPRETATIONS:")
print(f"• MAE of {current_mae:.6f} means predictions are off by ~{current_mae:.6f} units on average")
print(f"• R² of {overall_r2:.6f} means the model explains {accuracy_percentage:.2f}% of the variance")
print(f"• This indicates {interpret_r2_score(overall_r2).lower()}")

# Save results to a summary file
summary_results = {
    'mae': current_mae,
    'mse_loss': current_loss,
    'r2_score': overall_r2,
    'accuracy_percentage': accuracy_percentage,
    'individual_r2_scores': individual_r2_scores
}

import json
with open('../models/traffic_models/model_evaluation_results.json', 'w') as f:
    json.dump(summary_results, f, indent=2)

print("\nEvaluation results saved to '../models/traffic_models/model_evaluation_results.json'")
print("=" * 60)